Data exploration

In [ ]:
from pathlib import Path
import pandas as pd

import seaborn as sns
import matplotlib.pyplot as plt

import seaborn as sns

In [ ]:
import os

In [ ]:
import random

In [ ]:
DATA_DIR = Path(r"C:\Users\20223920\Documents\DS\25-26\Q4\Vak 1 4CBLW020 Addressing real-world crime and security problems with data science\Repro location\CBL_17_London\resources\data_full")

csv_files = list(DATA_DIR.rglob("*.csv"))

print(f"Amount of files found: {len(csv_files)}")

In [ ]:
for file in csv_files:

    print("\n" + "="*60)
    print(f"FILE: {file.name}")
    print("="*60)

    df = pd.read_csv(file, low_memory=False)

    print(df.head())
    print(df.info())

    if "crime_type" in df.columns:
        print(df["crime_type"].value_counts().head(10))

In [ ]:
stop_and_search_files = [
    file for file in csv_files
    if "stop-and-search" in file.name
]

street_files = [
    file for file in csv_files
    if "street" in file.name
]

print(len(stop_and_search_files), len(street_files))

In [ ]:
stop_and_search_dfs = []

for file in stop_and_search_files:

    df = pd.read_csv(file, low_memory=False)

    df.columns = (
        df.columns
        .str.lower()
        .str.replace(" ", "_")
        .str.replace("-", "_")
    )

    stop_and_search_dfs.append(df)

    print(file.name)
    print(list(df.columns))
    print(len(df.columns))

In [ ]:
expected_columns_sas = [
    'type',
    'date',
    'part_of_a_policing_operation',
    'policing_operation',
    'latitude',
    'longitude',
    'gender',
    'age_range',
    'self_defined_ethnicity',
    'officer_defined_ethnicity',
    'legislation',
    'object_of_search',
    'outcome',
    'outcome_linked_to_object_of_search',
    'removal_of_more_than_just_outer_clothing'
]

In [ ]:
consistent_files_sas = 0

for df in stop_and_search_dfs:

    column_names = list(df.columns)

    if column_names == expected_columns_sas:
        consistent_files_sas += 1

print(consistent_files_sas)

No inconsistent column names found

In [ ]:
street_dfs = []

for file in street_files:

    df = pd.read_csv(file, low_memory=False)

    df.columns = (
        df.columns
        .str.lower()
        .str.replace(" ", "_")
        .str.replace("-", "_")
    )

    street_dfs.append(df)

    print(file.name)
    print(list(df.columns))
    print(len(df.columns))

In [ ]:
expected_columns_street = [
    'crime_id',
    'month',
    'reported_by',
    'falls_within',
    'longitude',
    'latitude',
    'location',
    'lsoa_code',
    'lsoa_name',
    'crime_type',
    'last_outcome_category',
    'context'
]

In [ ]:
consistent_files_street = 0

for df in street_dfs:

    column_names = list(df.columns)

    if column_names == expected_columns_street:
        consistent_files_street += 1

print(consistent_files_street)

No inconsistent column names found

In [ ]:
street_df_full_raw = pd.concat(street_dfs, ignore_index=True)
street_df_full_raw.head()

In [ ]:
missing_matrix = street_df_full_raw.isna()

In [ ]:
sample = street_df_full_raw.sample(n=1000, random_state=42)
missing_matrix_sample = sample.isna()

In [ ]:
plt.figure(figsize=(12,6))
sns.heatmap(missing_matrix_sample, cbar=False)

plt.title("Missing values (sample of 1000 rows)")
plt.show()

In [ ]:
street_df_full_raw.isna().mean().sort_values(ascending=False).plot(kind="bar")

plt.title("Percentage missing values per column")
plt.ylabel("Fraction missing")
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
missing_counts = missing_matrix.sum()
missing_counts = missing_counts[missing_counts > 0]

print("Total rows:", len(street_df_full_raw))

for col, cnt in missing_counts.items():
    print(f"{col}: {cnt}")

In [ ]:
street_df_full_raw = street_df_full_raw.dropna(subset=["lsoa_code"])

In [ ]:
street_df_full_raw.head()

In [ ]:
crime_lsoa_month = street_df_full_raw.groupby(
    ["lsoa_code", "lsoa_name", "month"]
).size().reset_index(name="crime_count")

In [ ]:
crime_lsoa_month.head()

In [ ]:
street_df_full = street_df_full_raw.drop(
    columns=[
        "last_outcome_category",
        "context"
    ]
)

In [ ]:
street_df_full.head()

In [ ]:
street_df_full_raw["crime_type"].value_counts().head(40).plot(kind="bar")

plt.title("Distribution of amount of crimes per type")
plt.xticks(fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10,5))
sns.histplot(crime_lsoa_month["crime_count"], bins=100)

plt.xlim(0, 175)
plt.title("Distribution of crime counts per LSOA-month")
plt.xlabel("Crime count per LSOA-month")
plt.ylabel("Number of LSOA-months")

plt.show()

In [ ]:
crime_per_lsoa = crime_lsoa_month.groupby("lsoa_code")["crime_count"].sum()
top40 = crime_per_lsoa.sort_values(ascending=False).head(50)

plt.figure(figsize=(7,5))
top40.plot(kind="bar")

plt.title("Top 50 LSOAs by total crime")
plt.xlabel("LSOA")
plt.ylabel("Total number of crimes")

plt.xticks(rotation=45, fontsize=8)
plt.tight_layout()

plt.xticks([])  # removes x-axis labels

plt.show()

In [ ]:
monthly_total = crime_lsoa_month.groupby("month")["crime_count"].sum()

monthly_total.plot(figsize=(12,4))

plt.title("Total crime over time")
plt.xlabel("Month")
plt.ylabel("Total number of crimes")

plt.xticks(rotation=45)
plt.tight_layout()

plt.show()

In [ ]:
sample_lsoas = random.sample(
    list(crime_lsoa_month["lsoa_code"].unique()), 3
)

plt.figure(figsize=(12,3))

for lsoa in sample_lsoas:
    temp = crime_lsoa_month[crime_lsoa_month["lsoa_code"] == lsoa]
    temp = temp.sort_values("month")
    plt.plot(temp["month"], temp["crime_count"], label=lsoa)

plt.title("Crime trends for 3 random LSOAs")
plt.xlabel("Month")
plt.ylabel("Crime count")
plt.legend()
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
sample_lsoas = random.sample(
    list(crime_lsoa_month["lsoa_code"].unique()), 3
)

plt.figure(figsize=(12,3))

for lsoa in sample_lsoas:
    temp = crime_lsoa_month[crime_lsoa_month["lsoa_code"] == lsoa]
    temp = temp.sort_values("month")
    plt.plot(temp["month"], temp["crime_count"], label=lsoa)

plt.title("Crime trends for 3 random LSOAs")
plt.xlabel("Month")
plt.ylabel("Crime count")
plt.legend()
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
street_events = street_df_full.drop(
    columns=[
        "location",
        "longitude",
        "latitude",
        "lsoa_name",
        "reported_by"
    ]
)

In [ ]:
street_events.head()

In [ ]:
os.makedirs("../resources", exist_ok=True) 

street_events.to_parquet( 
    "../resources/street_events.parquet", 
    index=False, 
    engine="pyarrow" 
    ) 
crime_lsoa_month.to_parquet( 
    "../resources/crime_lsoa_month.parquet", 
    index=False, 
    engine="pyarrow" 
    )